In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import random
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [11]:
#data collection
data = pd.read_csv(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\Datasets\rideflow_datasets.csv")

In [12]:
#timestamp conversion to 15 mins bins
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['time_bin'] = data['timestamp'].dt.floor('15min')

In [13]:
#paths
output_dir = r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\Datasets\hotspot_images"
os.makedirs(output_dir, exist_ok=True)

In [14]:
#function defining for labels

# Calculate demand per time bin
demand_counts = data.groupby('time_bin').size()

# Dynamic thresholds
low_thresh = demand_counts.quantile(0.33)
high_thresh = demand_counts.quantile(0.66)

print("Low threshold:", low_thresh)
print("High threshold:", high_thresh)

def get_label(count):
    if count >= high_thresh:
        return "high"
    elif count >= low_thresh:
        return "medium"
    else:
        return "low"

Low threshold: 47.0
High threshold: 53.0


In [15]:
#
image_count = 0

for time_bin, subset in data.groupby('time_bin'):

    if len(subset) < 30:
        continue

    label = get_label(len(subset))

    heatmap, _, _ = np.histogram2d(
        subset['pickup_long'],
        subset['pickup_lat'],
        bins=64
    )

    heatmap = np.log1p(heatmap)

    plt.figure(figsize=(4,4))
    plt.imshow(heatmap, cmap='jet')
    plt.axis('off')

    folder = os.path.join(output_dir, label)
    os.makedirs(folder, exist_ok=True)

    img_path = os.path.join(folder, f"img_{image_count}.png")
    plt.savefig(img_path, bbox_inches='tight', pad_inches=0)
    plt.close()

    image_count += 1

print("Total Images:", image_count)

Total Images: 1000


In [16]:
#data split

split_dir = os.path.join(output_dir, "split")

for split in ["train", "val", "test"]:
    for cls in ["low", "medium", "high"]:
        os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

for cls in ["low", "medium", "high"]:

    cls_path = os.path.join(output_dir, cls)
    files = os.listdir(cls_path)
    random.shuffle(files)

    train_split = int(0.7 * len(files))
    val_split = int(0.85 * len(files))

    for f in files[:train_split]:
        shutil.copy(os.path.join(cls_path, f), os.path.join(split_dir, "train", cls, f))

    for f in files[train_split:val_split]:
        shutil.copy(os.path.join(cls_path, f), os.path.join(split_dir, "val", cls, f))

    for f in files[val_split:]:
        shutil.copy(os.path.join(cls_path, f), os.path.join(split_dir, "test", cls, f))

In [19]:
train_dir = os.path.join(split_dir, "train")
val_dir = os.path.join(split_dir, "val")
test_dir = os.path.join(split_dir, "test")

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_gen.flow_from_directory(train_dir, target_size=(224,224), batch_size=16, class_mode='categorical')
val_data = val_gen.flow_from_directory(val_dir, target_size=(224,224), batch_size=16, class_mode='categorical')
test_data = test_gen.flow_from_directory(test_dir, target_size=(224,224), batch_size=16, class_mode='categorical')

Found 698 images belonging to 3 classes.
Found 150 images belonging to 3 classes.
Found 152 images belonging to 3 classes.


In [20]:
#effiecient net
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(3, activation='softmax')(x)

eff_model = Model(inputs=base_model.input, outputs=output)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=4, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-6)
]

eff_model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

Epoch 1/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 28s 511ms/step - accuracy: 0.3223 - loss: 1.7061 - val_accuracy: 0.3000 - val_loss: 1.1650 - learning_rate: 3.0000e-04
Epoch 2/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 23s 525ms/step - accuracy: 0.3639 - loss: 1.5078 - val_accuracy: 0.2800 - val_loss: 1.1271 - learning_rate: 3.0000e-04
Epoch 3/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 549ms/step - accuracy: 0.4040 - loss: 1.4235 - val_accuracy: 0.3333 - val_loss: 1.1211 - learning_rate: 3.0000e-04
Epoch 4/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 546ms/step - accuracy: 0.3711 - loss: 1.4194 - val_accuracy: 0.3800 - val_loss: 1.1103 - learning_rate: 3.0000e-04
Epoch 5/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 540ms/step - accuracy: 0.3782 - loss: 1.4243 - val_accuracy: 0.3733 - val_loss: 1.1375 - learning_rate: 3.0000e-04
Epoch 6/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 23s 527ms/step - accuracy: 0.4241 - loss: 1.3326 - val_accuracy: 0.3800 - val_loss: 1.1303 - learning_rate: 3.0000e-04
Epoch 7/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 537ms/step - acc

In [21]:
for layer in base_model.layers[:-40]:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    layer.trainable = True

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

eff_model.fit(train_data, validation_data=val_data, epochs=10)

Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 52s 680ms/step - accuracy: 0.3238 - loss: 1.6467 - val_accuracy: 0.3267 - val_loss: 1.1366
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.3725 - loss: 1.4730 - val_accuracy: 0.3733 - val_loss: 1.1615
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.3596 - loss: 1.5623 - val_accuracy: 0.3800 - val_loss: 1.1837
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 33s 736ms/step - accuracy: 0.3582 - loss: 1.6020 - val_accuracy: 0.3600 - val_loss: 1.2070
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 40s 908ms/step - accuracy: 0.4112 - loss: 1.4803 - val_accuracy: 0.3267 - val_loss: 1.2330
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 29s 651ms/step - accuracy: 0.3653 - loss: 1.5741 - val_accuracy: 0.3067 - val_loss: 1.2550
Epoch 7/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 28s 631ms/step - accuracy: 0.3754 - loss: 1.5260 - val_accuracy: 0.3200 - val_loss: 1.2639
Epoch 8/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 28s 622ms/step - accuracy: 0.3610 - loss: 1.6087 - val_accuracy: 

In [22]:
loss, acc = eff_model.evaluate(test_data)
print("Final Test Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 305ms/step - accuracy: 0.3618 - loss: 1.2594
Final Test Accuracy: 0.3618420958518982
